# RetainIQ — Phase 2.3: Customer Flags & Integrity

## Objective

Add the approved analytical flag and prove that cleaning has not changed the customer-level
grain, customer identities, outcome relationship, or financial measures.

## 1. Load Raw Source

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_raw.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("telco_raw.csv")

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded raw dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")

Loaded raw dataset: 7,043 rows × 50 columns


## 2. Rebuild the Phase 2 Working Dataset

In [2]:
df_clean = df_raw.copy()

for c in df_clean.select_dtypes(include="object").columns:
    df_clean[c] = df_clean[c].str.strip()

df_clean["Offer"] = df_clean["Offer"].fillna("No Offer")
df_clean["Internet Type"] = df_clean["Internet Type"].fillna("No Internet Service")

print("Approved transformations applied.")

Approved transformations applied.


## 3. Create `is_new_customer`

In [3]:
df_clean["is_new_customer"] = df_clean["Customer Status"].eq("Joined")

df_clean["is_new_customer"].value_counts()

is_new_customer
False    6589
True      454
Name: count, dtype: int64

In [4]:
assert df_clean["is_new_customer"].dtype == bool
assert int(df_clean["is_new_customer"].sum()) == int(df_raw["Customer Status"].eq("Joined").sum())

print(f"PASS — {int(df_clean['is_new_customer'].sum()):,} new customers flagged.")

PASS — 454 new customers flagged.


## 4. Record-Level Logic Validation

In [5]:
expected_flag = df_clean["Customer Status"].eq("Joined")

assert df_clean["is_new_customer"].equals(expected_flag)

print("PASS — is_new_customer is exactly equivalent to Customer Status == 'Joined'.")

PASS — is_new_customer is exactly equivalent to Customer Status == 'Joined'.


## 5. Customer Grain Validation

In [6]:
grain_comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Unique Customer IDs",
        "Duplicate Customer IDs",
        "Missing Customer IDs"
    ],
    "Raw": [
        len(df_raw),
        df_raw["Customer ID"].nunique(),
        int(df_raw["Customer ID"].duplicated().sum()),
        int(df_raw["Customer ID"].isna().sum())
    ],
    "Clean": [
        len(df_clean),
        df_clean["Customer ID"].nunique(),
        int(df_clean["Customer ID"].duplicated().sum()),
        int(df_clean["Customer ID"].isna().sum())
    ]
})

grain_comparison

,Metric,Raw,Clean
0,Rows,7043,7043
1,Unique Customer IDs,7043,7043
2,Duplicate Customer IDs,0,0
3,Missing Customer IDs,0,0


In [7]:
assert len(df_clean) == len(df_raw)
assert df_clean["Customer ID"].is_unique
assert df_clean["Customer ID"].notna().all()
assert df_clean["Customer ID"].equals(df_raw["Customer ID"])

print("PASS — Customer-level grain preserved exactly.")

PASS — Customer-level grain preserved exactly.


## 6. Customer Outcome Integrity

In [8]:
raw_ct = pd.crosstab(df_raw["Customer Status"], df_raw["Churn Label"])
clean_ct = pd.crosstab(df_clean["Customer Status"], df_clean["Churn Label"])

pd.concat({"Raw": raw_ct, "Clean": clean_ct}, axis=1)

Raw       Clean      
Churn Label        No   Yes    No   Yes
Customer Status                        
Churned             0  1869     0  1869
Joined            454     0   454     0
Stayed           4720     0  4720     0

In [9]:
assert raw_ct.equals(clean_ct)
print("PASS — Customer Status × Churn Label relationship preserved.")

PASS — Customer Status × Churn Label relationship preserved.


## 7. Financial Field Integrity

In [10]:
financial_cols = [
    "Monthly Charge",
    "Total Charges",
    "Total Refunds",
    "Total Extra Data Charges",
    "Total Long Distance Charges",
    "Total Revenue",
    "CLTV"
]

integrity = pd.DataFrame({
    "Field": financial_cols,
    "Raw Missing": [df_raw[c].isna().sum() for c in financial_cols],
    "Clean Missing": [df_clean[c].isna().sum() for c in financial_cols],
    "Raw Negative": [(df_raw[c] < 0).sum() for c in financial_cols],
    "Clean Negative": [(df_clean[c] < 0).sum() for c in financial_cols]
})

integrity

,Field,Raw Missing,Clean Missing,Raw Negative,Clean Negative
0,Monthly Charge,0,0,0,0
1,Total Charges,0,0,0,0
2,Total Refunds,0,0,0,0
3,Total Extra Data Charges,0,0,0,0
4,Total Long Distance Charges,0,0,0,0
5,Total Revenue,0,0,0,0
6,CLTV,0,0,0,0


In [11]:
for c in financial_cols:
    assert df_clean[c].equals(df_raw[c])

print("PASS — Financial values are unchanged.")

PASS — Financial values are unchanged.


## 8. Schema Change Audit

In [12]:
added = sorted(set(df_clean.columns) - set(df_raw.columns))
removed = sorted(set(df_raw.columns) - set(df_clean.columns))

pd.DataFrame({
    "Change Type": ["Added", "Removed"],
    "Columns": [added, removed]
})

,Change Type,Columns
0,Added,[is_new_customer]
1,Removed,[]


In [13]:
assert added == ["is_new_customer"]
assert removed == []
print("PASS — Only the approved analytical flag was added.")

PASS — Only the approved analytical flag was added.


## 9. Notebook 3 Conclusion

The cleaned working dataset retains one record per customer, preserves all original financial
values and outcome relationships, and adds exactly one approved analytical field.

**Next:** `04_cleaning_validation_and_export.ipynb`